# Independent phase-centered preprocessing and sex provenance

**Scientific purpose.** Reproduce the phase-specific image preprocessing that generated retained
internal model inputs and preserve the corrected clinical-variable provenance boundary.

**Runnability classification:** runnable from user-obtained PLC-CECT V4 images, private split
definitions, and configured private output storage.

**Inputs:** phase-specific CT, tumor-mask, and liver-mask NIfTI files plus the private cohort split.
**Private outputs:** one compressed patient-level array per patient.

Each phase is independently centered in its own voxel grid. The four `96 x 96 x 96` arrays are
stacked only after cropping. No explicit interphase registration, physical-coordinate mapping, or
additional internal resampling was performed.

Corrected clinical handling maps male to 1, female to 0, and retains unresolved sex as missing.
The historical exact-token rule that mapped every non-male entry to zero is stored separately only
for provenance sensitivity analysis.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from liver_tumor_pipeline.config import load_config, require_path

CONFIG_PATH = REPO_ROOT / "configs" / "paths.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        "Create an untracked configs/paths.yaml from configs/paths.example.yaml and "
        "set the documented environment variables before running this workflow."
    )

CONFIG = load_config(CONFIG_PATH)
PRIVATE_ARTIFACT_ROOT = require_path(CONFIG, "private_artifact_root", must_exist=False)
OUTPUT_ROOT = require_path(CONFIG, "output_root", must_exist=False)

import json
import numpy as np
import pandas as pd
import nibabel as nib

from liver_tumor_pipeline.preprocessing import preprocess_independent_phases

INTERNAL_ROOT = require_path(CONFIG, "plc_cect_root")
MANIFEST_PATH = INTERNAL_ROOT / "patient_data.csv"
SPLIT_PATH = PRIVATE_ARTIFACT_ROOT / "splits" / "train_val_test.json"
PROCESSED_ROOT = PRIVATE_ARTIFACT_ROOT / "processed" / "internal"
for required in (MANIFEST_PATH, SPLIT_PATH):
    if not required.is_file():
        raise FileNotFoundError(
            "Required PLC-CECT manifest or private split definition is unavailable."
        )


## Loader and patient adapter

The loader preserves the retained array convention, including reduction of singleton and redundant
RGB-like trailing dimensions. Affine/header geometry is not used by the crop algorithm and was not
retained in cached arrays. Sex encoding preserves unresolved values for the corrected primary path.


In [ ]:
PHASE_ORDER = ("P", "C1", "C2", "C3")
LABEL_MAP = {"HCC": 0, "ICC": 1, "CHCC": 2, "cHCC-CCA": 2}


def load_historical_array(path: Path) -> np.ndarray:
    array = nib.load(str(path)).get_fdata().astype(np.float32)
    while array.ndim > 3:
        if array.shape[-1] == 1:
            array = array.squeeze(-1)
        elif array.shape[-1] == 3:
            array = array.mean(axis=-1)
        else:
            array = array[..., 0]
    if array.ndim != 3:
        raise ValueError(f"Expected a 3D array after dimension reduction, got {array.shape}")
    return array


def resolve_dataset_path(relative_or_absolute: str) -> Path:
    path = Path(relative_or_absolute)
    return path if path.is_absolute() else INTERNAL_ROOT / path


def corrected_sex_value(raw_value) -> np.float32:
    if pd.isna(raw_value):
        return np.float32(np.nan)
    normalized = str(raw_value).strip().lower()
    if normalized == "male":
        return np.float32(1.0)
    if normalized == "female":
        return np.float32(0.0)
    return np.float32(np.nan)


def historical_sex_value(raw_value) -> np.int64:
    return np.int64(1 if str(raw_value).lower() == "male" else 0)


def process_patient(patient_rows: pd.DataFrame) -> dict[str, np.ndarray | float | int | str]:
    patient_values = patient_rows["patient_id"].astype(str).unique()
    if len(patient_values) != 1:
        raise ValueError("Patient adapter received zero or multiple patient identifiers.")
    patient_id = patient_values[0]
    by_phase = patient_rows.set_index("phase")
    missing_phases = sorted(set(PHASE_ORDER).difference(by_phase.index))
    if missing_phases:
        raise ValueError(f"Patient is missing required phases: {missing_phases}")

    ct_by_phase = {}
    tumor_by_phase = {}
    liver_by_phase = {}
    for phase in PHASE_ORDER:
        row = by_phase.loc[phase]
        ct_by_phase[phase] = load_historical_array(resolve_dataset_path(row["ct_path"]))
        tumor_by_phase[phase] = load_historical_array(resolve_dataset_path(row["mask_path"]))
        liver_by_phase[phase] = load_historical_array(resolve_dataset_path(row["liver_mask_path"]))

    crops = preprocess_independent_phases(
        ct_by_phase,
        tumor_by_phase,
        liver_by_phase,
        crop_size=96,
    )
    first = patient_rows.iloc[0]
    class_token = str(first["cancer_type"])
    if class_token not in LABEL_MAP:
        raise ValueError(f"Unsupported class token: {class_token}")
    raw_sex = first.get("gender")
    age_value = float(first["age"]) if pd.notna(first.get("age")) else np.nan
    return {
        "ct": crops["ct"],
        "tumor_mask": crops["tumor_mask"],
        "liver_mask": crops["liver_mask"],
        "label": np.int64(LABEL_MAP[class_token]),
        "patient_id": patient_id,
        "age": np.float32(age_value),
        "gender": corrected_sex_value(raw_sex),
        "historical_gender": historical_sex_value(raw_sex),
        "tumor_vol_portal": np.float32(tumor_by_phase["C2"].sum()),
    }


## Private preprocessing run

Patient-linked filenames and arrays stay outside the repository. This stage records
only aggregate completion counts in the notebook interface. It does not create a
distributable archive.


In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
required_columns = {
    "patient_id", "phase", "cancer_type", "ct_path", "mask_path", "liver_mask_path", "gender"
}
missing_columns = sorted(required_columns.difference(manifest.columns))
if missing_columns:
    raise ValueError(f"Internal manifest is missing required columns: {missing_columns}")

split = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
patients_to_process = sorted({str(value) for value in split["dev_ids"] + split["test_ids"]})
if len(patients_to_process) != 278:
    raise RuntimeError("Locked preprocessing cohort must contain 278 unique patients.")

PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
completed = 0
failed = []
for patient_id in patients_to_process:
    patient_rows = manifest[manifest["patient_id"].astype(str) == patient_id]
    try:
        payload = process_patient(patient_rows)
        if payload["ct"].shape != (4, 96, 96, 96):
            raise RuntimeError("Four-phase crop shape invariant failed.")
        np.savez_compressed(PROCESSED_ROOT / f"{patient_id}.npz", **payload)
        completed += 1
    except Exception as error:
        failed.append(type(error).__name__)

if failed:
    raise RuntimeError(
        f"Preprocessing failed for {len(failed)} patients; inspect private source artifacts."
    )
pd.DataFrame({"status": ["completed"], "patients": [completed]})


## Interpretation boundary

The public provenance audit found 185 male, 90 female, and three unresolved entries. One unresolved
entry was in development and two were in the held-out set. Corrected fusion fitting uses fold-local
median imputation; the `historical_gender` field exists only to reproduce the approved sensitivity.

The fixed crop truncated a subset of C2 tumors. Equivalent retention audits for P, C1, and C3
cannot be reconstructed because source geometry and crop coordinates were not retained.
